In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT_DIR = Path("/content/drive/MyDrive/nlpcc")
DATA_DIR = ROOT_DIR / "data"
OUTPUTS_DIR = ROOT_DIR / "outputs"
DEV_FILE = DATA_DIR / "dev.jsonl"
EVAL_OUT_DIR = OUTPUTS_DIR / 'track2' / 'eval'
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

# 选择 Track2 生成模型
TRACK2_MODEL_NAME = 'dpo'
TRACK2_MODEL_DIR = ROOT_DIR / 'outputs' / 'track2' / TRACK2_MODEL_NAME
TRACK2_BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

TRACK1_MODEL_DIR = OUTPUTS_DIR / 'track1' / 'encoder' / 'bert_scl_optimized'
TRACK1_BASE_MODEL = 'bert-base-uncased'

MAX_PROMPT_LENGTH = 768
MAX_NEW_TOKENS = 96
TRACK1_MAX_LENGTH = 512
TRACK1_BATCH_SIZE = 16

# 如果 generation_file 已存在，就直接读取，避免重复生成。
USE_CACHE = True

GENERATION_FILE = EVAL_OUT_DIR / f'{TRACK2_MODEL_NAME}_dev_generations.jsonl'
DETAIL_FILE = EVAL_OUT_DIR / f'{TRACK2_MODEL_NAME}_track1_eval_details.csv'
print('DEV_FILE:', DEV_FILE)
print('TRACK2_MODEL_DIR:', TRACK2_MODEL_DIR)
print('TRACK1_MODEL_DIR:', TRACK1_MODEL_DIR)
print('GENERATION_FILE:', GENERATION_FILE)

DEV_FILE: /content/drive/MyDrive/nlpcc/data/dev.jsonl
TRACK2_MODEL_DIR: /content/drive/MyDrive/nlpcc/outputs/track2/dpo
TRACK1_MODEL_DIR: /content/drive/MyDrive/nlpcc/outputs/track1/encoder/bert_scl_optimized
GENERATION_FILE: /content/drive/MyDrive/nlpcc/outputs/track2/eval/dpo_dev_generations.jsonl


In [4]:
# Cell 2: Official values, IO, prompt builder

VALUE_DEFINITIONS = {
    'Self-direction–thought': "Freedom to cultivate one's own ideas and abilities.",
    'Self-direction–action': "Freedom to determine one's own actions.",
    'Stimulation': 'Excitement, novelty, and change.',
    'Hedonism': 'Pleasure and sensuous gratification.',
    'Achievement': 'Success according to social standards.',
    'Power–dominance': 'Power through exercising control over people.',
    'Power–resources': 'Power through control of material and social resources.',
    'Face': "Security and power through maintaining one's public image and avoiding humiliation.",
    'Security–personal': "Safety in one's immediate environment.",
    'Security–societal': 'Safety and stability in the wider society.',
    'Tradition': 'Maintaining and preserving cultural, family, or religious traditions.',
    'Conformity–rules': 'Compliance with rules, laws, and formal obligations.',
    'Conformity–interpersonal': 'Avoidance of upsetting or harming other people.',
    'Humility': "Recognizing one's insignificance in the larger scheme of things.",
    'Benevolence–dependability': 'Being a reliable and trustworthy member of the ingroup.',
    'Benevolence–caring': 'Devotion to the welfare of ingroup members.',
    'Universalism–concern': 'Commitment to equality, justice, and protection for all people.',
    'Universalism–nature': 'Preservation of the natural environment.',
    'Universalism–tolerance': 'Acceptance and understanding of those who are different from oneself.',
}
VALUE_LABELS = list(VALUE_DEFINITIONS.keys())
label2id = {v: i for i, v in enumerate(VALUE_LABELS)}
id2label = {i: v for i, v in enumerate(VALUE_LABELS)}

TASK_INSTRUCTION = (
    'You are given a scenario, a question, and a target human value. '
    'Generate one concise, meaningful response that answers the question, '
    'fits the scenario, and naturally aligns with the target value.'
)
def normalize_space(text):
    return re.sub(r'\s+', ' ', str(text or '')).strip()

def build_prompt(row, use_value_definition=True):
    scenario = normalize_space(row.get('Scenario', ''))
    question = normalize_space(row.get('Question', ''))
    value = normalize_space(row.get('Value', ''))
    parts = [
        TASK_INSTRUCTION,
        f'Scenario:\n{scenario}',
        f'Question:\n{question}',
        f'Target value:\n{value}',
    ]
    if use_value_definition:
        parts.append(f'Target value definition:\n{VALUE_DEFINITIONS.get(value, "")}')
    parts.append('Response:')
    return '\n\n'.join(parts) + '\n'

def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
dev_rows = read_jsonl(DEV_FILE)
for i, row in enumerate(dev_rows):
  row['idx'] = i
  row["prompt"] = build_prompt(row)
print("dev rows: ", len(dev_rows))
print('sample prompt:\n', dev_rows[0]['prompt'][:700])


dev rows:  514
sample prompt:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
A teammate insists on frequent communication to feel secure.

Question:
How would you handle the teammate’s demand for constant updates?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:



In [6]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.7 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [9]:
# Cell 3 : Generate Track2 response
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def load_track2_model(model_dir):
  model_dir = Path(model_dir)
  tokenizer_scr = str(model_dir) if model_dir.exists() else TRACK2_BASE_MODEL
  tokenizer = AutoTokenizer.from_pretrained(tokenizer_scr, trust_remote_code = True)
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
  if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

  model_kwargs = {'trust_remote_code': True}
  if torch.cuda.is_available():
    model_kwargs['torch_dtype'] = torch.bfloat16
    model_kwargs['device_map'] = 'auto'
  if (model_dir / 'adapter_config.json').exists():
    model = AutoModelForCausalLM.from_pretrained(TRACK2_BASE_MODEL, **model_kwargs)
    model.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(model, str(model_dir))
  model.config.pad_token_id = tokenizer.pad_token_id
  model.eval()
  return model, tokenizer

def generate_response(model, tokenizer, prompt):
  device = next(model.parameters()).device
  inputs = tokenizer(prompt, return_tensors = 'pt', truncation = True, max_length=MAX_PROMPT_LENGTH)
  inputs = {k : v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens = MAX_NEW_TOKENS,
        do_sample = False,
        pad_token_id = tokenizer.pad_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )
  gen_ids = output[0, inputs['input_ids'].shape[1]:]
  return normalize_space(tokenizer.decode(gen_ids, skip_special_tokens=True))

if USE_CACHE and GENERATION_FILE.exists():
  generated_rows = read_jsonl(GENERATION_FILE)
  print('Loaded cached generations:', GENERATION_FILE)
else:
  model, tokenizer = load_track2_model(TRACK2_MODEL_DIR)
  generated_rows = []
  for row in tqdm(dev_rows):
    generated_rows.append({
        'idx':row['idx'],
        'Value':row['Value'],
        'row_response': row['Consistent Value Response'],
        'generated_response': generate_response(model, tokenizer, row['prompt']),
    })
  write_jsonl(GENERATION_FILE, generated_rows)
  print('Saved generations:', GENERATION_FILE)

print('generated size:', len(generated_rows))
print(generated_rows[0])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  0%|          | 0/514 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Saved generations: /content/drive/MyDrive/nlpcc/outputs/track2/eval/dpo_dev_generations.jsonl
generated size: 514
{'idx': 0, 'Value': 'Conformity–interpersonal', 'row_response': 'I would prioritize maintaining that frequent contact despite my schedule, as I want to ensure my teammate feels valued, included, and that our teamwork remains perfectly smooth.', 'generated_response': 'I will offer gentle reassurance while avoiding any actions that might make them feel uncomfortable, ensuring their sense of belonging is maintained without causing distress.'}


In [11]:
# Cell 4: Classify generated responses with Track1 model
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

def load_track1_model(model_dir):
  tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code = True)
  common_kwargs = dict(
      num_labels = len(VALUE_LABELS),
      id2label = id2label,
      label2id = label2id,
      trust_remote_code = True,
  )
  if (model_dir / 'adpater_config.json').exists():
    model = AutoModelForSequenceClassification.from_pretrained(TRACK1_BASE_MODEL, **common_kwargs)
    model = PeftModel.from_pretrained(model, str(model_dir))
  else:
    model = AutoModelForSequenceClassification.from_pretrained(str(model_dir), **common_kwargs)

  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  model.to(device)
  model.eval()
  return model, tokenizer, device
track1_model, track1_tokenizer, track1_device = load_track1_model(TRACK1_MODEL_DIR)

def classify_batch(texts):
  inputs = track1_tokenizer(
      ['Response: ' + normalize_space(t) for t in texts],
      padding = True,
      truncation = True,
      max_length = TRACK1_MAX_LENGTH,
      return_tensors = 'pt',
  )
  inputs = {k : v.to(track1_device) for k,v in inputs.items()}
  with torch.no_grad():
    logits = track1_model(**inputs).logits.float()
    probs = torch.softmax(logits, dim = -1).cpu().numpy()
  return probs

response = [row['generated_response'] for row in generated_rows]
all_probs = []
for start in tqdm(range(0, len(response), TRACK1_BATCH_SIZE)):
  all_probs.append(classify_batch(response[start:start + TRACK1_BATCH_SIZE]))
probs = np.concatenate(all_probs, axis = 0)

records = []
for row, gen, prob in zip(dev_rows, generated_rows, probs):
  pred_id = int(prob.argmax())
  target_id = label2id[row['Value']]
  sorted_prob = np.sort(prob)
  records.append({
      'idx':row['idx'],
      'Value':row['Value'],
      'pred_value':id2label[pred_id],
      'is_match' : id2label[pred_id] == row['Value'],
      'track1_target_prob': float(prob[target_id]),
      'track1_pred_prob': float(prob[pred_id]),
      'Question': row['Question'],
      'reference_response': row['Consistent Value Response'],
      'generated_response': gen['generated_response'],
  })
eval_df = pd.DataFrame(records)
eval_df.to_csv(DETAIL_FILE, index=False)
print('Saved details:', DETAIL_FILE)
display(eval_df.head())

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight       

Loading weights: 0it [00:00, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: /content/drive/MyDrive/nlpcc/outputs/track1/encoder/bert_scl_optimized
Key                                                                                             | Status     | 
------------------------------------------------------------------------------------------------+------------+-
base_model.model.deberta.encoder.layer.{0...23}.attention.self.value_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.deberta.encoder.layer.{0...23}.attention.output.dense.lora_A.default.weight    | UNEXPECTED | 
base_model.model.deberta.encoder.layer.{0...23}.attention.self.query_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.deberta.encoder.layer.{0...23}.attention.self.key_proj.lora_B.default.weight   | UNEXPECTED | 
base_model.model.deberta.encoder.layer.{0...23}.attention.self.value_proj.lora_B.default.weight | UNEXPECTED | 
base_model.model.deberta.encoder.layer.{0...23}.attention.self.key_proj.lora_A.default.weigh

  0%|          | 0/33 [00:00<?, ?it/s]

Saved details: /content/drive/MyDrive/nlpcc/outputs/track2/eval/dpo_track1_eval_details.csv


,idx,Value,pred_value,is_match,track1_target_prob,track1_pred_prob,Question,reference_response,generated_response
0,0,Conformity–interpersonal,Power–dominance,False,0.056286,0.090970,How would you handle the teammate’s demand for...,I would prioritize maintaining that frequent c...,I will offer gentle reassurance while avoiding...
1,1,Conformity–interpersonal,Power–dominance,False,0.044631,0.084417,How would you handle a situation where your id...,I would remain quiet and simply listen to the ...,I will try to incorporate their feedback while...
2,2,Conformity–interpersonal,Power–dominance,False,0.035491,0.083879,Would you voice your concerns about the plan e...,I would choose to remain silent or perhaps add...,I will gently bring up my thoughts while makin...
3,3,Conformity–interpersonal,Power–dominance,False,0.044144,0.086504,How would you respond to a coworker's public c...,I would approach my colleague privately to exp...,I will gently but respectfully address the sit...
4,4,Conformity–interpersonal,Power–dominance,False,0.056136,0.088779,How would you handle the decision-making proce...,I would pursue compromise solutions that minim...,I will carefully consider how my actions affec...


In [15]:
# Cell 5: Results

summary = pd.DataFrame([{
    'model': TRACK2_MODEL_NAME,
    'n': len(eval_df),
    'track1_match_rate': eval_df['is_match'].mean(),
    'avg_target_prob': eval_df['track1_target_prob'].mean(),
    'avg_pred_prob': eval_df['track1_pred_prob'].mean(),
}]).round(4)

display(summary)

by_value = (
    eval_df
    .groupby('Value')
    .agg(
        n=('idx', 'count'),
        match_rate=('is_match', 'mean'),
        avg_target_prob=('track1_target_prob', 'mean'),
    )
    .reset_index()
    .sort_values('match_rate')
    .round(4)
)
display(by_value)

summary.to_csv(EVAL_OUT_DIR / f'{TRACK2_MODEL_NAME}_summary.csv', index=False)
by_value.to_csv(EVAL_OUT_DIR / f'{TRACK2_MODEL_NAME}_by_value.csv', index=False)

,model,n,track1_match_rate,avg_target_prob,avg_pred_prob
0,dpo,514,0.0214,0.0528,0.0971


,Value,n,match_rate,avg_target_prob
0,Achievement,26,0.0000,0.0457
1,Benevolence–caring,46,0.0000,0.0593
2,Benevolence–dependability,27,0.0000,0.0663
3,Conformity–interpersonal,34,0.0000,0.0490
4,Conformity–rules,55,0.0000,0.0418
5,Face,37,0.0000,0.0564
6,Hedonism,24,0.0000,0.0577
9,Power–resources,35,0.0000,0.0481
14,Stimulation,58,0.0000,0.0505
11,Security–societal,11,0.0000,0.0463
